In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
JupyterDash.infer_jupyter_proxy_config()

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "Swifyfunf"

# Connect to database via CRUD Module
db = AnimalShelter()

# class read method must support return of list object and accept projection json input
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png'
import base64
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Hr(),

    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode())),

    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {"label": "Water Rescue", "value": "Water"},
                {"label": "Mountain Rescue", "value": "Mountain"},
                {"label": "Disaster Rescue", "value": "Disaster"},
                {"label": "Reset", "value": "Reset"}
            ],
            value="Reset"
        )
    ]),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True}
                 for i in df.columns],
        data=df.to_dict('records'),
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        row_selectable="single",
        page_size=10
    ),

    html.Br(),
    html.Hr(),

    html.Div(className='row', style={'display': 'flex'}, children=[

        html.Div(id='graph-id', className='col s12 m6'),

        html.Div(id='map-id', className='col s12 m6')

    ])
])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output('datatable-id', "data"),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    if filter_type == "Water":
        df = pd.DataFrame.from_records(db.read({
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Newfoundland"]},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 104}
        }))

    elif filter_type == "Mountain":
        df = pd.DataFrame.from_records(db.read({
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Bernese Mountain Dog"]},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 104}
        }))

    elif filter_type == "Disaster":
        df = pd.DataFrame.from_records(db.read({
            "animal_type": "Dog",
            "breed": {"$in": ["Bloodhound", "German Shepherd", "Doberman Pinscher"]},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 104}
        }))

    else:
        df = pd.DataFrame.from_records(db.read({}))

    if '_id' in df.columns:
        df.drop(columns=['_id'], inplace=True)

    return df.to_dict('records')


@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):

    if viewData is None:
        return

    dff = pd.DataFrame.from_dict(viewData)

    return [
        dcc.Graph(
            figure=px.pie(dff, names='breed', title='Breed Distribution')
        )
    ]


@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, selected_rows):

    if viewData is None:
        return

    dff = pd.DataFrame.from_dict(viewData)

    row = 0 if not selected_rows else selected_rows[0]

    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
               center=[30.75, -97.48],
               zoom=10,
               children=[
                   dl.TileLayer(),
                   dl.Marker(
                       position=[dff.iloc[row]['location_lat'],
                                 dff.iloc[row]['location_long']],
                       children=[
                           dl.Tooltip(dff.iloc[row]['breed']),
                           dl.Popup([
                               html.H1("Animal Name"),
                               html.P(dff.iloc[row]['name'])
                           ])
                       ]
                   )
               ])
    ]


# Run app
app.run_server(mode='jupyterlab', port=8054)